# Wind Power Forecasting: LLM Evaluation

Based on the original team's working implementation.

**Run cells in order from top to bottom.**

## 1. Import Libraries

In [ ]:
import os
import re
import json
import time
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import google.generativeai as genai
import anthropic

print('✅ Libraries imported')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('wtbdata_245days.csv')
print(f'✅ Loaded {len(df)} rows from SDWPF dataset')

## 3. Configure API Keys and Models

In [ ]:
# API Keys
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
CLAUDE_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')

# Configure clients
genai.configure(api_key=GEMINI_API_KEY)
claude_client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)

# Model configurations
models = [
    {"provider": "gemini", "name": "gemini-2.5-flash", "label": "Gemini 2.5 Flash"},
    {"provider": "claude", "name": "claude-haiku-4-5-20251001", "label": "Claude 4.5 Haiku"},
    {"provider": "gemini", "name": "gemini-2.5-pro", "label": "Gemini 2.5 Pro"},
    {"provider": "claude", "name": "claude-sonnet-4-5-20250929", "label": "Claude 4.5 Sonnet"}
]

print('✅ Configured 4 models:')
for m in models:
    print(f"   - {m['label']}")

## 4. Helper Functions

In [ ]:
def bin_turbine_data(df, n_bins=16, bin_cols=None):
    """Bins numeric columns into equal-width integer IDs [0, n_bins-1]"""
    hist = df.copy()
    if bin_cols is None:
        excluded = {"TurbID", "Day", "Tmstamp"}
        bin_cols = [
            c for c in hist.columns 
            if (c not in excluded) and pd.api.types.is_numeric_dtype(hist[c])
        ]

    for c in bin_cols:
        x = pd.to_numeric(hist[c], errors="coerce")
        valid = x.dropna()
        if valid.empty:
            continue

        lo, hi = float(valid.min()), float(valid.max())
        
        if np.isclose(lo, hi):
            hist[c] = 0
            continue

        edges = np.linspace(lo, hi, n_bins + 1)
        binned = pd.cut(x, bins=edges, labels=False, include_lowest=True, duplicates="drop")
        hist[c] = binned.astype("Int64")
        
    return hist, bin_cols


def get_openmeteo_wind_data(lat, lon, start_date, end_date):
    """Fetches 100m wind speed from Open-Meteo Archive API"""
    import requests
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "wind_speed_100m",
        "wind_speed_unit": "ms"
    }
    
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    
    if 'hourly' not in data or 'wind_speed_100m' not in data['hourly']:
        raise ValueError(f"Unexpected API response format")
    
    timestamps = pd.to_datetime(data['hourly']['time'])
    wind_speeds = data['hourly']['wind_speed_100m']
    
    return pd.DataFrame({
        'timestamp': timestamps,
        'wind_speed_100m': wind_speeds
    })


def evaluate_forecast(df, turb_id, base_day, output, horizon_steps):
    """Evaluates forecast for specified horizon"""
    target_start = base_day + 14
    days_needed = int(np.ceil(horizon_steps / 144))
    target_end = target_start + days_needed - 1
    
    actual_data = df[
        (df['TurbID'] == turb_id) &
        (df['Day'] >= target_start) &
        (df['Day'] <= target_end)
    ]
    
    actual_values = actual_data.sort_values(['Day', 'Tmstamp'])['Patv'].dropna().tolist()
    
    # Parse JSON 
    try:
        clean_json = output.replace("```json", "").replace("```", "").strip()
        data = json.loads(clean_json)
        predicted_values = data.get("forecast", [])
    except Exception as e:
        print(f"   ❌ JSON parsing failed: {e}")
        return None
    
    min_len = min(len(actual_values), len(predicted_values), horizon_steps)
    
    if min_len == 0:
        print(f"   ❌ No overlapping data")
        return None
    
    y_true = actual_values[:min_len]
    y_pred = predicted_values[:min_len]
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    return {
        "MAE": mae,
        "RMSE": rmse,
        "points_used": min_len
    }


print('✅ Helper functions loaded')

## 5. LLM Calling Functions (Students' Retry Logic)

In [ ]:
def call_llm_naive(df, turb_id, base_day, model_config, horizon_steps):
    """Naive Baseline: Simple prompt with JSON output and retry logic"""
    
    turbine_data = df[df['TurbID'] == turb_id].dropna(subset=['Wspd'])
    end_day = base_day + 13

    window_data = turbine_data[
        (turbine_data['Day'] >= base_day) &
        (turbine_data['Day'] <= end_day)
    ].drop(columns=['TurbID'])

    data_str = window_data.to_csv(index=False, sep=',')
    horizon_hours = horizon_steps * 10 / 60

    prompt = f"""You are given 14 days of historical SCADA data for ONE turbine.
The data is sampled every 10 minutes (144 rows per day).

Input Data:
{data_str}

Your task:
Predict the Active Power (Patv, in kW) for the NEXT {horizon_hours:.1f} HOURS ({horizon_steps} time steps at 10-minute resolution).

OUTPUT FORMAT REQUIREMENTS:
Return valid JSON in the following format:
{{
  "forecast": [f1, f2, f3, ..., f{horizon_steps}]
}}

The list MUST contain exactly {horizon_steps} numbers.
No text outside the JSON.
No explanations.
No markdown formatting.
Only raw JSON.
"""

    # RETRY LOGIC (Students' approach)
    MAX_LLM_ATTEMPTS = 3
    final_output = None

    for attempt in range(1, MAX_LLM_ATTEMPTS + 1):
        print(f"   Attempt {attempt}/{MAX_LLM_ATTEMPTS}...", end=" ")
        
        try:
            # Generate response
            if model_config['provider'] == 'claude':
                response = claude_client.messages.create(
                    model=model_config['name'],
                    max_tokens=max(8192, horizon_steps * 30),
                    messages=[{"role": "user", "content": prompt}]
                )
                raw_text = response.content[0].text
            else:  # Gemini
                model = genai.GenerativeModel(model_config['name'])
                response = model.generate_content(prompt)
                raw_text = response.text

            # Validate JSON
            clean_json = raw_text.replace('```json', '').replace('```', '').strip()
            data = json.loads(clean_json)
            
            if "forecast" in data and len(data["forecast"]) == horizon_steps:
                print(f"✅ Got {horizon_steps} values")
                final_output = raw_text
                break
            else:
                actual_len = len(data.get("forecast", []))
                print(f"❌ Wrong length: {actual_len} (expected {horizon_steps})")
        
        except Exception as e:
            print(f"❌ Error: {str(e)[:50]}")

        if attempt < MAX_LLM_ATTEMPTS:
            time.sleep(2)

    if final_output is None:
        print("   ❌ All attempts failed")
        return None

    return final_output


def call_llm_apbf(df, turb_id, base_day, model_config, horizon_steps, era5_lat, era5_lon):
    """APBF: Advanced Prompt + Binning + Forecast (Students' approach)"""
    
    turbine_data = df[df['TurbID'] == turb_id].dropna(subset=['Wspd'])
    end_day = base_day + 13

    window_data = turbine_data[
        (turbine_data['Day'] >= base_day) &
        (turbine_data['Day'] <= end_day)
    ].copy()

    # BINNING (Students' approach)
    window_data, binned_list = bin_turbine_data(window_data, n_bins=16, bin_cols=["Wspd", "Patv"])
    bin_msg = f"Note: Numeric columns ({', '.join(binned_list)}) have been binned into 16 equal-width levels (0-15)."
    data_str = window_data.to_csv(index=False, sep=',')

    # ERA5 FORECAST
    anchor_date = datetime(2020, 5, 1)
    forecast_start_date = anchor_date + timedelta(days=int(base_day + 13))
    days_needed = int(np.ceil(horizon_steps / 144))
    forecast_end_date = forecast_start_date + timedelta(days=days_needed)

    start_str = forecast_start_date.strftime('%Y-%m-%d')
    end_str = forecast_end_date.strftime('%Y-%m-%d')

    try:
        weather_df = get_openmeteo_wind_data(era5_lat, era5_lon, start_str, end_str)
        w_speeds = weather_df['wind_speed_100m'].values
        xp = np.arange(len(w_speeds))
        x_new = np.linspace(0, len(w_speeds)-1, horizon_steps)
        interp_w_speeds = np.interp(x_new, xp, w_speeds)
        forecast_str = ", ".join([f"{v:.2f}" for v in interp_w_speeds])
        forecast_section = f"""Input 2: Meteorological Forecast (100m wind)
Predicted wind speeds (m/s) at 10-minute intervals for the next {horizon_steps * 10 / 60:.1f} hours:
[{forecast_str}]"""
    except Exception as e:
        print(f"   ⚠️ ERA5 fetch failed: {e}")
        forecast_section = "No meteorological forecast available."

    horizon_hours = horizon_steps * 10 / 60

    prompt = f"""Context: You are an expert wind turbine power forecasting model.

{bin_msg}

Input 1: You are given 14 days of historical SCADA data for ONE turbine.
The data is sampled every 10 minutes (144 rows per day).
{data_str}

{forecast_section}

Your task:
Predict the Active Power (Patv, in kW) for the NEXT {horizon_hours:.1f} HOURS ({horizon_steps} time steps at 10-minute resolution).

Instructions:
- Learn the wind-speed to power relationship from the historical data
- Power is roughly proportional to wind speed cubed at low speeds
- Power saturates near rated power (~1500 kW)
- Capture daily cyclic patterns
- Do NOT output negative values
- Clip values to the realistic range [0, 1500]

OUTPUT FORMAT REQUIREMENTS (CRITICAL):
Return valid JSON in the following format:
{{
  "forecast": [f1, f2, f3, ..., f{horizon_steps}]
}}

The list MUST contain exactly {horizon_steps} numbers.
If the list contains more or fewer values, the answer is invalid.

No text outside the JSON.
No explanations.
No markdown formatting.
Only raw JSON.
"""

    # RETRY LOGIC (Students' approach)
    MAX_LLM_ATTEMPTS = 3
    final_output = None

    for attempt in range(1, MAX_LLM_ATTEMPTS + 1):
        print(f"   Attempt {attempt}/{MAX_LLM_ATTEMPTS}...", end=" ")
        
        try:
            # Generate response
            if model_config['provider'] == 'claude':
                response = claude_client.messages.create(
                    model=model_config['name'],
                    max_tokens=max(8192, horizon_steps * 30),
                    messages=[{"role": "user", "content": prompt}]
                )
                raw_text = response.content[0].text
            else:  # Gemini
                model = genai.GenerativeModel(model_config['name'])
                response = model.generate_content(prompt)
                raw_text = response.text

            # Validate JSON
            clean_json = raw_text.replace('```json', '').replace('```', '').strip()
            data = json.loads(clean_json)
            
            if "forecast" in data and len(data["forecast"]) == horizon_steps:
                print(f"✅ Got {horizon_steps} values")
                final_output = raw_text
                break
            else:
                actual_len = len(data.get("forecast", []))
                print(f"❌ Wrong length: {actual_len} (expected {horizon_steps})")
        
        except Exception as e:
            print(f"❌ Error: {str(e)[:50]}")

        if attempt < MAX_LLM_ATTEMPTS:
            time.sleep(2)

    if final_output is None:
        print("   ❌ All attempts failed")
        return None

    return final_output


print('✅ LLM calling functions loaded (with retry logic)')

## 6. Run Experiments

In [ ]:
# Configuration
turb_id = 1
base_day = 36
WIND_FARM_LATITUDE = 40.6306
WIND_FARM_LONGITUDE = 96.9498

# Horizons
horizons = [
    {"name": "3h", "steps": 18},
    {"name": "6h", "steps": 36},
    {"name": "48h", "steps": 288}
]

# Strategies
strategies = [
    {"name": "Naive", "func": call_llm_naive},
    {"name": "APBF", "func": call_llm_apbf}
]

# Run experiments
results = []
total_experiments = len(models) * len(strategies) * len(horizons)
current = 0

print(f"\n🚀 Starting {total_experiments} experiments...\n")
print("=" * 80)

for model_cfg in models:
    for strategy in strategies:
        for horizon in horizons:
            current += 1
            exp_name = f"{model_cfg['label']} | {strategy['name']} | {horizon['name']}"
            print(f"\n[{current}/{total_experiments}] {exp_name}")
            print("-" * 80)
            
            try:
                # Call LLM with retry logic
                if strategy['name'] == 'Naive':
                    output = strategy['func'](df, turb_id, base_day, model_cfg, horizon['steps'])
                else:  # APBF
                    output = strategy['func'](
                        df, turb_id, base_day, model_cfg, horizon['steps'],
                        era5_lat=WIND_FARM_LATITUDE, era5_lon=WIND_FARM_LONGITUDE
                    )
                
                if output is None:
                    print("   ⚠️ Skipping experiment (all retry attempts failed)")
                    continue
                
                # Evaluate
                eval_result = evaluate_forecast(df, turb_id, base_day, output, horizon['steps'])
                
                if eval_result:
                    results.append({
                        "Model": model_cfg['label'],
                        "Strategy": strategy['name'],
                        "Horizon": horizon['name'],
                        "MAE": round(eval_result['MAE'], 2),
                        "RMSE": round(eval_result['RMSE'], 2)
                    })
                    print(f"   ✅ MAE: {eval_result['MAE']:.2f} kW | RMSE: {eval_result['RMSE']:.2f} kW")
                else:
                    print("   ❌ Evaluation failed")
                    
            except Exception as e:
                print(f"   ❌ Error: {e}")
            
            time.sleep(1)

print("\n" + "=" * 80)
print(f"✅ Completed {len(results)}/{total_experiments} experiments")
print("=" * 80)

## 7. Results Table

In [ ]:
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Save to CSV
results_df.to_csv('wind_forecasting_results.csv', index=False)
print('\n✅ Results saved to wind_forecasting_results.csv')

## 8. Generate LaTeX Table

In [ ]:
# Generate LaTeX table
latex_lines = []
latex_lines.append(r"\begin{table}[h]")
latex_lines.append(r"\centering")
latex_lines.append(r"\caption{Wind Power Forecasting: LLM Performance vs Naive Baseline}")
latex_lines.append(r"\label{tab:wind_results}")
latex_lines.append(r"\begin{tabular}{llcccccc}")
latex_lines.append(r"\toprule")
latex_lines.append(r"\textbf{Model} & \textbf{Strategy} & \multicolumn{2}{c}{\textbf{3h}} & \multicolumn{2}{c}{\textbf{6h}} & \multicolumn{2}{c}{\textbf{48h}} \\")
latex_lines.append(r"\cmidrule(lr){3-4} \cmidrule(lr){5-6} \cmidrule(lr){7-8}")
latex_lines.append(r" & & MAE & RMSE & MAE & RMSE & MAE & RMSE \\")
latex_lines.append(r"\midrule")

for model_label in results_df['Model'].unique():
    # Get baseline (Naive) results
    naive_results = {}
    naive_subset = results_df[(results_df['Model'] == model_label) & (results_df['Strategy'] == 'Naive')]
    for _, row in naive_subset.iterrows():
        naive_results[row['Horizon']] = {'MAE': row['MAE'], 'RMSE': row['RMSE']}
    
    # Naive row
    naive_3h_mae = f"{naive_results.get('3h', {}).get('MAE', 0):.1f}" if '3h' in naive_results else "--"
    naive_3h_rmse = f"{naive_results.get('3h', {}).get('RMSE', 0):.1f}" if '3h' in naive_results else "--"
    naive_6h_mae = f"{naive_results.get('6h', {}).get('MAE', 0):.1f}" if '6h' in naive_results else "--"
    naive_6h_rmse = f"{naive_results.get('6h', {}).get('RMSE', 0):.1f}" if '6h' in naive_results else "--"
    naive_48h_mae = f"{naive_results.get('48h', {}).get('MAE', 0):.1f}" if '48h' in naive_results else "--"
    naive_48h_rmse = f"{naive_results.get('48h', {}).get('RMSE', 0):.1f}" if '48h' in naive_results else "--"
    
    latex_lines.append(
        f"{model_label} & Naive & {naive_3h_mae} & {naive_3h_rmse} & "
        f"{naive_6h_mae} & {naive_6h_rmse} & {naive_48h_mae} & {naive_48h_rmse} \\\\"
    )
    
    # APBF row with improvement percentages
    apbf_subset = results_df[(results_df['Model'] == model_label) & (results_df['Strategy'] == 'APBF')]
    apbf_results = {}
    for _, row in apbf_subset.iterrows():
        apbf_results[row['Horizon']] = {'MAE': row['MAE'], 'RMSE': row['RMSE']}
    
    # Calculate improvements
    apbf_vals = {}
    for hz in ['3h', '6h', '48h']:
        if hz in apbf_results and hz in naive_results:
            mae_val = apbf_results[hz]['MAE']
            mae_baseline = naive_results[hz]['MAE']
            mae_improv = ((mae_baseline - mae_val) / mae_baseline * 100) if mae_baseline > 0 else 0
            
            rmse_val = apbf_results[hz]['RMSE']
            rmse_baseline = naive_results[hz]['RMSE']
            rmse_improv = ((rmse_baseline - rmse_val) / rmse_baseline * 100) if rmse_baseline > 0 else 0
            
            apbf_vals[hz] = {
                'mae': f"{mae_val:.1f} ({mae_improv:+.1f}\\%)",
                'rmse': f"{rmse_val:.1f} ({rmse_improv:+.1f}\\%)"
            }
        else:
            apbf_vals[hz] = {'mae': '--', 'rmse': '--'}
    
    latex_lines.append(
        f"{model_label} & APBF & {apbf_vals['3h']['mae']} & {apbf_vals['3h']['rmse']} & "
        f"{apbf_vals['6h']['mae']} & {apbf_vals['6h']['rmse']} & "
        f"{apbf_vals['48h']['mae']} & {apbf_vals['48h']['rmse']} \\\\"
    )
    
    latex_lines.append(r"\midrule")

latex_lines[-1] = r"\bottomrule"
latex_lines.append(r"\end{tabular}")
latex_lines.append(r"\end{table}")

latex_table = "\n".join(latex_lines)
print(latex_table)

with open('results_table.tex', 'w') as f:
    f.write(latex_table)
    
print('\n✅ LaTeX table saved to results_table.tex')